# LightCyto: three-dataset visualization and download notebook

**SIPaKMeD · Cervical Cytology (four-class) · CRIC (binary)**

Use this notebook **after training and saved validation/final-test evaluation**. It reads existing reports only. It does not train, rerun inference, select a model or optimize a threshold. A CPU session is sufficient; a GPU is not required.

**Input:** the reports ZIP or exact full output folder from each dataset's LightCyto notebook. Raw image datasets, screenshots and model weights alone are not sufficient. Missing datasets are clearly marked and skipped.

**Output:** separate PNG (300 DPI), vector PDF and editable SVG figures; CSV metrics/curve coordinates; an offline HTML gallery; one ZIP per available dataset; and one combined download ZIP.

The implementation is plain Python in the cells below—no compressed or encoded code payload.


## 1. Attach these three saved outputs

Your screenshots show these experiment folders:

| Dataset | Kaggle input label shown | Exact experiment folder |
|---|---|---|
| SIPaKMeD | My_final_model | `lightcyto_tinyvit_end_to_end` |
| Cervical Cytology | Dataset 2 | `lightcyto_cervical_cytology_ablation` |
| CRIC | dataset_3 | `lightcyto_cric_ablation` |

Keep all three outputs attached. **Run All**: the notebook locates those exact folder names anywhere below `/kaggle/input`, including `/kaggle/input/datasets/<owner>/<dataset>/...` layouts. Display labels need not match the mounted dataset slug.

The screenshots identify folders, not their full contents. The preflight cell checks the actual attached files before any rendering. For Cervical Cytology, `final_test` is not visible in the screenshot; if test predictions are absent, only available training/validation results are plotted.


In [ ]:
from pathlib import Path
import importlib.util
required = ['numpy', 'pandas', 'matplotlib', 'sklearn']
missing = [m for m in required if importlib.util.find_spec(m) is None]
if missing:
    raise RuntimeError(f"Missing packages: {missing}. Install numpy pandas matplotlib scikit-learn, then restart/run again.")
print('Dependencies available. No GPU or model weights needed.')


## 2. Paths already configured for your screenshots

Leave each entry as **`AUTO`** for the attached outputs shown above. No need to guess whether Kaggle converted a display label to underscores or hyphens.

If more than one copy/version of an exact experiment folder is attached, the notebook lists the candidates and asks for an exact path in `SOURCES`. It never silently selects the newest or mixes experiments. You can also override an entry with the exact reports ZIP path, or set it to `SKIP` deliberately.


In [ ]:
SOURCES = {
    'SIPaKMeD': 'AUTO',
    'Cervical_Cytology': 'AUTO',
    'CRIC': 'AUTO',
}
SOURCE_FOLDER_NAMES = {
    'SIPaKMeD': 'lightcyto_tinyvit_end_to_end',
    'Cervical_Cytology': 'lightcyto_cervical_cytology_ablation',
    'CRIC': 'lightcyto_cric_ablation',
}
SEARCH_ROOTS = [Path('/kaggle/input')] if Path('/kaggle/input').exists() else [Path.cwd() / 'inputs']
OUTPUT_ROOT = Path('/kaggle/working/lightcyto_visualizations') if Path('/kaggle/working').exists() else Path.cwd() / 'lightcyto_visualizations'
FORMATS = ('png', 'pdf', 'svg')  # PNG is required for the gallery
DPI = 300


## 3. What is generated

| Available input | Plots / tables |
|---|---|
| Training history | Training objective, CE/KD components if logged; validation accuracy, macro-F1, balanced accuracy and CE; learning rate and time; PPO entropy/KL/crop use when logged |
| Split summary | Image and grouping-unit counts by class and split |
| Per-image predictions | Count/true-normalized/predicted-normalized confusion matrices, precision, recall/sensitivity, specificity, F1, NPV, support, overall accuracy, balanced accuracy, macro/weighted/micro F1, MCC, kappa, error pairs and group diagnostics |
| Class probabilities | ROC/AUROC, precision–recall/AP, reliability/ECE, log loss, Brier score, confidence/entropy histograms, risk–coverage, mean-probability matrix, curve-coordinate CSVs |
| Multiple arms/seeds | Accuracy/F1 comparisons, individual-seed points and sample SD, paired error counts, validation versus test comparisons (not a time curve) |
| Saved uncertainty tables | Bootstrap CI and paired-effect plots using the existing resampling protocol |
| Saved efficiency tables | Parameter/size/compute/latency plots and accuracy–latency scatter |
| Saved PPO predictions | Crop count/subset distributions and paired GLOBAL/PPO errors |
| Saved attention images | Original attention examples copied into `saved_visuals/` |

**Interpretation:** unprefixed `accuracy`, `macro_f1` and `ce` in the supplied LightCyto training logs are **validation** values. If training accuracy was never logged, no training accuracy curve is invented. Train loss can include smoothing/KD and differs from validation CE.

The final test set is evaluated at the locked checkpoint. It has **ROC/PR/calibration curves**, not an invented epoch-wise test curve. Do not evaluate it every epoch merely to draw a curve.

Grad-CAM, new attention overlays, t-SNE/UMAP of learned features, representative image grids and robustness sweeps need images/features/checkpoints. This report-only notebook marks those limitations instead of manufacturing explanations from prediction probabilities.


In [ ]:
"""LightCyto report-only visualization exporter. No training or test inference."""
from pathlib import Path
import json, re, hashlib, zipfile, shutil, html, itertools, textwrap, warnings, io, os, fnmatch
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn import metrics as skm

VERSION = '1.1.0-path-configured'
plt.rcParams.update({'font.family':'DejaVu Sans','font.size':10,'axes.spines.top':False,
 'axes.spines.right':False,'axes.titleweight':'bold','axes.titlesize':13,
 'axes.labelsize':10,'savefig.facecolor':'white','svg.fonttype':'none','pdf.fonttype':42})
COLORS = ['#007F86','#D17830','#4C62A8','#A5426A','#698C45','#7D6097','#435F70']
plt.rcParams['axes.prop_cycle'] = plt.cycler(color=COLORS)
ARM_NAMES = {'A_PLAIN':'Plain TinyViT','B_KD':'TinyViT + KD',
 'C_ATTENTION':'TinyViT + attention','D_FULL':'TinyViT + attention + KD'}

def slug(s): return re.sub(r'[^A-Za-z0-9_.-]+','_',str(s)).strip('_')
def short(s):
    return {'Superficial-Intermediate':'Superficial / intermediate',
      'High squamous intra-epithelial lesion':'HSIL',
      'Low squamous intra-epithelial lesion':'LSIL',
      'Negative for Intraepithelial malignancy':'NILM',
      'Squamous cell carcinoma':'SCC'}.get(str(s),str(s))
def wrap(s, n=24): return '\n'.join(textwrap.wrap(short(s),n))
def readj(p): return json.loads(Path(p).read_text())
def sha(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for b in iter(lambda:f.read(1024*1024),b''): h.update(b)
    return h.hexdigest()
def writej(p, data):
    Path(p).parent.mkdir(parents=True, exist_ok=True)
    Path(p).write_text(json.dumps(data,indent=2,default=str,allow_nan=False))
def numeric(df,c): return pd.to_numeric(df[c],errors='coerce')


# Ignore nested exported copies so the same predictions are not counted twice.
CONTRACT_MARKERS = ('experiment.json', 'ablation_contract.json')
def report_files(root, pattern='*'):
    root=Path(root)
    for current, dirs, files in os.walk(root, followlinks=False):
        current=Path(current)
        kept=[]
        for name in dirs:
            child=current/name
            nested_experiment=any((child/marker).is_file() for marker in CONTRACT_MARKERS)
            exported_copy=name.lower().startswith('lightcyto_') and name.lower().endswith(('_reports','_checkpoints'))
            if not nested_experiment and not exported_copy and name not in ('__MACOSX','_extracted'):
                kept.append(name)
        dirs[:]=kept
        for name in sorted(files):
            if fnmatch.fnmatch(name,pattern):yield current/name


### Plot and metric exporter

In [ ]:
class Exporter:
    def __init__(self,name,source,out,formats=('png','pdf','svg'),dpi=300):
        self.name=name; self.root=Path(source); self.out=Path(out)
        self.out.mkdir(parents=True,exist_ok=True)
        self.formats=formats; self.dpi=dpi; self.index=[]; self.notes=[]; self.metrics=[]; self.runs=[]
    def note(self,area,message): self.notes.append({'area':area,'message':message})
    def table(self,rel,df,index=False):
        p=self.out/(rel+'.csv');p.parent.mkdir(parents=True,exist_ok=True);df.to_csv(p,index=index)
    def fig(self,rel,fig,title,note=''):
        fig.suptitle(title,fontsize=13,fontweight='bold',y=1.01)
        if note: fig.text(.01,-.015,note,fontsize=8,color='#52606B',wrap=True)
        fig.tight_layout()
        p=self.out/rel;p.parent.mkdir(parents=True,exist_ok=True)
        for ext in self.formats:
            buffer=io.BytesIO()
            fig.savefig(buffer,format=ext,dpi=self.dpi,bbox_inches='tight')
            payload=buffer.getvalue()
            if not payload: raise RuntimeError('Empty rendered figure: '+str(p))
            target=Path(str(p)+'.'+ext); temporary=Path(str(target)+'.tmp')
            temporary.write_bytes(payload); temporary.replace(target)
        plt.close(fig); self.index.append({'plot':rel,'title':title,'note':note})
    def heat(self,rel,arr,rows,cols,title,fmt='.1f',cmap='Blues',vmin=None,vmax=None,note=''):
        a=np.asarray(arr,dtype=float);fig,ax=plt.subplots(figsize=(max(6,len(cols)*1.05+2),max(4.6,len(rows)*.65+1.8)))
        im=ax.imshow(np.ma.masked_invalid(a),cmap=cmap,vmin=vmin,vmax=vmax,aspect='auto')
        ax.set_xticks(range(len(cols)),[wrap(x,18) for x in cols],rotation=35,ha='right')
        ax.set_yticks(range(len(rows)),[wrap(x) for x in rows])
        maxv=np.nanmax(a) if np.isfinite(a).any() else 1
        for i in range(a.shape[0]):
            for j in range(a.shape[1]):
                t='NA' if not np.isfinite(a[i,j]) else format(a[i,j],fmt)
                ax.text(j,i,t,ha='center',va='center',fontsize=9,color='white' if a[i,j]>maxv*.58 else '#102A43')
        fig.colorbar(im,ax=ax,shrink=.8)
        self.fig(rel,fig,title,note)
    def copy_metadata(self):
        records=[]
        for p in sorted(report_files(self.root,'*')):
            if p.is_file() and p.suffix.lower() in ('.csv','.json'):
                rel=p.relative_to(self.root); target=self.out/'source_tables'/rel;target.parent.mkdir(parents=True,exist_ok=True)
                shutil.copy2(p,target); records.append({'path':str(rel),'bytes':p.stat().st_size,'sha256':sha(p)})
        self.table('source_file_inventory',pd.DataFrame(records))
        for p in report_files(self.root,'attention_examples.png'):
            dest=self.out/'saved_visuals'/p.relative_to(self.root);dest.parent.mkdir(parents=True,exist_ok=True);shutil.copy2(p,dest)
        for p in self.root.glob('architecture.*'):
            if p.suffix in ('.png','.svg','.pdf'):
                dest=self.out/'saved_visuals'/p.name;dest.parent.mkdir(parents=True,exist_ok=True);shutil.copy2(p,dest)
    def distribution(self):
        paths=list(report_files(self.root,'split_summary.csv'))
        if len(paths)!=1:
            self.note('dataset distribution','Need one split_summary.csv with split, class_name and images.');return
        df=pd.read_csv(paths[0])
        if not {'split','class_name','images'}.issubset(df):return
        for value in ('images','groups'):
            if value not in df:continue
            t=df.pivot_table(index='class_name',columns='split',values=value,aggfunc='sum')
            t=t.reindex(columns=[c for c in ('train','val','validation','test') if c in t.columns])
            fig,ax=plt.subplots(figsize=(9,5));t.rename(index=short).plot.barh(ax=ax,color=COLORS[:len(t.columns)])
            ax.set(xlabel=value.capitalize(),ylabel='');ax.legend(title='Split',loc='best')
            self.fig('dataset/'+value+'_by_class_split',fig,f'{self.name} | {value} by class and split',
                'Group counts are reported grouping units; patient independence is not implied.' if value=='groups' else '')
            self.table('dataset/'+value+'_by_class_split',t,index=True)
        self.note('dataset scope','Read source_tables/data_audit.json and experiment/ablation contract for exclusions, grouping and prior test use.')
    def histories(self):
        paths=sorted(report_files(self.root,'history.csv'))
        if not paths:self.note('training curves','No history.csv files supplied. Cannot reconstruct epoch curves from final metrics.')
        for p in paths:
            df=pd.read_csv(p); rel=p.parent.relative_to(self.root).as_posix();base='learning_curves/'+slug(rel)
            if 'epoch' not in df and 'round' not in df:
                self.note(rel,'No epoch/round column.');continue
            xcol='epoch' if 'epoch' in df else 'round';x=numeric(df,xcol)
            if x.duplicated().any():raise ValueError(f'Duplicate epochs in {p}; resolve resumed history before plotting.')
            order=np.argsort(x.to_numpy());df=df.iloc[order];x=numeric(df,xcol)
            isppo='ppo' in rel.lower();xlabel='PPO round' if isppo else 'Epoch'
            groups=[('train_objective',['train_loss','train_ce','train_kd','train_rkd'],'Training objective (not validation CE)'),
                    ('validation_scores',['accuracy','macro_f1','balanced_accuracy','val_accuracy','val_macro_f1'],'Validation score (%)'),
                    ('validation_loss',['ce','val_loss','val_ce'],'Validation cross-entropy'),
                    ('training_scores',['train_accuracy','train_macro_f1'],'Training score (%)'),
                    ('learning_rate',['lr'],'Learning rate'),('runtime',['seconds'],'Seconds per epoch / round')]
            if isppo:groups += [('policy_entropy',['entropy'],'Policy entropy'),('approx_kl',['approx_kl'],'Approximate KL'),
                              ('validation_crop_use',['mean_crops'],'Validation mean extra crops'),('reward',['mean_reward','reward'],'Training reward')]
            for key,cols,ylabel in groups:
                cols=[c for c in cols if c in df and numeric(df,c).notna().any()]
                if not cols:continue
                fig,ax=plt.subplots(figsize=(8,4.6));ispercent=key.endswith('scores')
                for c in cols:ax.plot(x,numeric(df,c)*(100 if ispercent else 1),label=c.replace('_',' '),lw=1.8)
                ax.set(xlabel=xlabel,ylabel=ylabel);ax.grid(alpha=.2);ax.legend()
                note='Training components are logged unweighted; total loss includes configured weights.' if key=='train_objective' and 'train_kd' in cols else ''
                if key=='train_objective' and isppo:note='PPO optimization loss can be negative; it is not classification loss or reward.'
                self.fig(base+'/'+key,fig,f'{self.name} | {rel}',note)
            if 'train_accuracy' not in df:self.note(rel,'Training accuracy was not logged; only genuine logged training losses are plotted.')
            if not any(c.startswith('test_') for c in df):self.note(rel,'No epoch-wise test series. Final-test ROC/PR/calibration curves are provided instead.')
    def history_comparisons(self):
        histories=[]
        for p in sorted(report_files(self.root,'history.csv')):
            parts=p.relative_to(self.root).parts
            arm=next((v for v in parts if v in ARM_NAMES),None)
            seed=next((v[5:] for v in parts if re.fullmatch(r'seed_\d+',v)),None)
            if arm is None or seed is None:continue
            t=pd.read_csv(p)
            if 'epoch' not in t:continue
            t=t.assign(arm=arm,seed=seed);histories.append(t)
        if not histories:return
        allh=pd.concat(histories,ignore_index=True)
        if allh.duplicated(['arm','seed','epoch']).any():
            self.note('learning comparisons','Multiple stage histories per arm/seed/epoch. Individual curves retained; aggregate skipped.');return
        for metric,percent in [('train_loss',False),('accuracy',True),('macro_f1',True),('ce',False)]:
            if metric not in allh:continue
            agg=allh.groupby(['arm','epoch'])[metric].agg(['mean','std','count']).reset_index()
            self.table('learning_comparison/'+metric,agg)
            fig,ax=plt.subplots(figsize=(8,4.8))
            for i,(arm,t) in enumerate(agg.groupby('arm')):
                factor=100 if percent else 1;x=t.epoch.to_numpy();mean=t['mean'].to_numpy()*factor
                ax.plot(x,mean,label=ARM_NAMES.get(arm,arm),color=COLORS[i%len(COLORS)])
                if (t['count']>=2).all():
                    sd=t['std'].to_numpy()*factor
                    ax.fill_between(x,mean-sd,mean+sd,color=COLORS[i%len(COLORS)],alpha=.12)
            role='Training objective' if metric=='train_loss' else 'Validation '+metric.replace('_',' ')
            ax.set(xlabel='Epoch',ylabel=role+(' (%)' if percent else ''));ax.grid(alpha=.2);ax.legend(fontsize=8)
            self.fig('learning_comparison/'+metric,fig,self.name+' | '+role,
                'Lines: mean of available seeds at each epoch. Bands: sample seed SD when >=2; see count column for incomplete runs.')
    def identity(self,p):
        parts=p.relative_to(self.root).parts
        split='test' if 'final_test' in parts or 'test' in parts else 'validation' if any('validation'==v or 'val'==v for v in parts) else None
        seed=next((v[5:] for v in parts if re.fullmatch(r'seed_\d+',v)),'unspecified')
        arm=next((v for v in parts if v in ARM_NAMES),None)
        if arm is None: arm=p.parent.name
        if split is None:return None
        return split,arm,seed
    def parse_predictions(self,p):
        df=pd.read_csv(p); pc=[c for c in df if c.startswith('p_')]
        if not {'sample_id','label','prediction'}.issubset(df):raise ValueError(f'{p}: missing sample_id, label or prediction.')
        if df.sample_id.isna().any() or df.sample_id.duplicated().any():raise ValueError(f'{p}: sample IDs missing or duplicated.')
        if not len(df):raise ValueError(f'{p}: empty predictions.')
        # p_<class> column order is the label order in the LightCyto writers.
        if len(pc)>=2:names=[c[2:] for c in pc]
        else:
            mapping={}
            for c,n in [('label','class_name'),('prediction','predicted_class')]:
                if n in df:
                    for label,g in df.groupby(c):
                        if g[n].nunique()!=1:raise ValueError('Ambiguous class names.')
                        mapping[int(label)]=str(g[n].iloc[0])
            if not mapping:raise ValueError('No probabilities or verified class-name mapping.')
            names=[mapping[i] for i in range(max(mapping)+1)]
        y=numeric(df,'label').to_numpy();pred=numeric(df,'prediction').to_numpy();k=len(names)
        if not(np.isfinite(y).all() and np.isfinite(pred).all() and np.equal(y,y.astype(int)).all() and np.equal(pred,pred.astype(int)).all()):raise ValueError('Invalid class IDs.')
        y=y.astype(int);pred=pred.astype(int)
        if min(y.min(),pred.min())<0 or max(y.max(),pred.max())>=k:raise ValueError('Class IDs do not match probability columns.')
        if 'class_name' in df and not np.all(df.class_name.astype(str).to_numpy()==np.array(names)[y]):raise ValueError('Probability column order differs from true class IDs.')
        if 'predicted_class' in df and not np.all(df.predicted_class.astype(str).to_numpy()==np.array(names)[pred]):raise ValueError('Prediction names disagree with IDs.')
        probs=None
        if pc:
            probs=df[pc].to_numpy(float)
            if not np.isfinite(probs).all() or (probs<0).any() or (probs>1).any() or not np.allclose(probs.sum(1),1,atol=2e-5):raise ValueError('Invalid probability values or row sums.')
            probs=probs/probs.sum(1,keepdims=True) # tiny CSV rounding only
            if not np.array_equal(pred,probs.argmax(1)):raise ValueError('Saved predictions are not argmax probabilities; custom threshold requires a dedicated adapter.')
        return df,y,pred,probs,names
    def prediction_plots(self,p):
        identity=self.identity(p)
        if identity is None:self.note(str(p.relative_to(self.root)),'Unknown evaluation split; skipped.');return
        split,arm,seed=identity;df,y,pred,probs,names=self.parse_predictions(p);k=len(names);n=len(y)
        base=f'{split}/{slug(arm)}/seed_{seed}';title=f'{self.name} | {ARM_NAMES.get(arm,arm)} | {split} | seed {seed}'
        cm=skm.confusion_matrix(y,pred,labels=range(k));correct=y==pred
        support=cm.sum(1);predn=cm.sum(0)
        rows=np.divide(cm,support[:,None],out=np.full_like(cm,np.nan,dtype=float),where=support[:,None]!=0)
        cols=np.divide(cm,predn[None,:],out=np.full_like(cm,np.nan,dtype=float),where=predn[None,:]!=0)
        for suffix,a,fmt in [('counts',cm,'.0f'),('true_normalized_percent',100*rows,'.1f'),('predicted_normalized_percent',100*cols,'.1f')]:
            self.heat(base+'/confusion_'+suffix,a,names,names,title+'\nConfusion matrix — '+suffix.replace('_',' '),fmt,
                      note=f'Rows: true class. Columns: predicted class. n={n}; no seed pooling.')
            self.table(base+'/confusion_'+suffix,pd.DataFrame(a,index=names,columns=names),index=True)
        tp=np.diag(cm);fp=predn-tp;fn=support-tp;tn=n-tp-fp-fn
        div=lambda a,b:np.divide(a,b,out=np.full(k,np.nan),where=b!=0)
        prec,rec,f1,sup=skm.precision_recall_fscore_support(y,pred,labels=range(k),zero_division=0)
        classes=pd.DataFrame({'class':names,'precision':prec,'recall_sensitivity':rec,'f1':f1,'specificity_ovr':div(tn,tn+fp),
           'npv_ovr':div(tn,tn+fn),'support':sup,'predicted_count':predn,'TP':tp,'FP':fp,'FN':fn,'TN':tn})
        self.table(base+'/class_metrics',classes)
        self.heat(base+'/class_metric_heatmap',classes[['precision','recall_sensitivity','f1','specificity_ovr']].values*100,
             names,['Precision','Recall','F1','Specificity (OvR)'],title+'\nClass metrics (%)',vmin=0,vmax=100)
        m={'dataset':self.name,'split':split,'arm':arm,'seed':seed,'n':n,'correct':int(correct.sum()),'classes':k,
           'accuracy':float(correct.mean()),'balanced_accuracy':float(skm.balanced_accuracy_score(y,pred)),
           'macro_precision':float(prec.mean()),'macro_recall':float(rec.mean()),'macro_f1':float(f1.mean()),
           'weighted_f1':float(skm.f1_score(y,pred,average='weighted',zero_division=0)),
           'micro_f1':float(skm.f1_score(y,pred,average='micro',zero_division=0)),
           'mcc':float(skm.matthews_corrcoef(y,pred)), 'cohen_kappa':float(skm.cohen_kappa_score(y,pred))}
        errors=df.loc[~correct].copy().sort_values('confidence',ascending=False) if 'confidence' in df else df.loc[~correct].copy()
        self.table(base+'/errors',errors)
        pairs=[{'true_class':names[i],'predicted_class':names[j],'count':int(cm[i,j])} for i in range(k) for j in range(k) if i!=j]
        pairs=pd.DataFrame(pairs).sort_values('count',ascending=False);self.table(base+'/confusion_pairs',pairs)
        pairs=pairs[pairs['count']>0]
        if len(pairs):
            fig,ax=plt.subplots(figsize=(8,max(3,len(pairs)*.36)))
            ax.barh([short(a)+' → '+short(b) for a,b in zip(pairs.true_class,pairs.predicted_class)],pairs['count']);ax.invert_yaxis();ax.set_xlabel('Errors')
            self.fig(base+'/error_pairs',fig,title+'\nMisclassification pairs')
        if 'group_id' in df:
            gt=df[['group_id']].assign(correct=correct).groupby('group_id').agg(n=('correct','size'),correct=('correct','sum'))
            gt['accuracy']=gt.correct/gt.n;self.table(base+'/group_accuracy',gt,index=True)
            fig,ax=plt.subplots(figsize=(7,4));ax.scatter(gt.n,100*gt.accuracy,alpha=.65);ax.set(xlabel='Images in grouping unit',ylabel='Accuracy (%)',ylim=(-2,102))
            self.fig(base+'/group_accuracy',fig,title+'\nAccuracy by reported grouping unit','Descriptive; a grouping unit is not necessarily a patient.')
        if probs is not None:self.probability_plots(base,title,y,pred,probs,names,m)
        else:self.note(base,'No class probabilities: ROC, PR, calibration, log-loss and confidence plots unavailable.')
        if 'extra_crops' in df:
            crop=pd.to_numeric(df.extra_crops,errors='raise');m['mean_extra_crops']=float(crop.mean())
            c=crop.value_counts().sort_index();fig,ax=plt.subplots(figsize=(6,4));ax.bar(c.index.astype(str),c.values);ax.set(xlabel='Extra crops',ylabel='Images')
            self.fig(base+'/crop_count_distribution',fig,title+'\nCrop use',f'Mean extra crops: {crop.mean():.4f}')
            self.table(base+'/crop_count_distribution',c.rename_axis('extra_crops').reset_index(name='images'))
        if 'crop_subset_bits' in df:
            labels={0:'STOP / global',1:'Darkness',2:'Colour',3:'Darkness + colour',4:'Boundary',5:'Darkness + boundary',6:'Colour + boundary',7:'All three'}
            c=df.crop_subset_bits.value_counts().sort_index();fig,ax=plt.subplots(figsize=(8,4));ax.barh([labels.get(int(i),str(i)) for i in c.index],c.values);ax.set_xlabel('Images')
            self.fig(base+'/crop_subsets',fig,title+'\nFinal crop subsets','Subset counts do not reveal the order of PPO actions.')
        self.metrics.append(m);self.runs.append({'split':split,'arm':arm,'seed':seed,'df':df,'y':y,'pred':pred,'names':names})
    def probability_plots(self,base,title,y,pred,p,names,m):
        k=len(names);n=len(y);onehot=np.eye(k)[y];correct=y==pred;conf=p.max(1)
        m['log_loss']=float(skm.log_loss(y,p,labels=range(k)))
        m['brier_multiclass_sum']=float(np.mean(np.sum((p-onehot)**2,axis=1)))
        if k==2:m['brier_binary_label_1']=float(np.mean((p[:,1]-y)**2))
        rocfig,ra=plt.subplots(figsize=(7,5));prfig,pa=plt.subplots(figsize=(7,5));aucs=[];aps=[];curves=[]
        for i,name in enumerate(names):
            yi=(y==i).astype(int)
            if len(np.unique(yi))<2:continue
            fpr,tpr,thr=skm.roc_curve(yi,p[:,i]);auc=skm.auc(fpr,tpr)
            precision,recall,th=skm.precision_recall_curve(yi,p[:,i]);ap=skm.average_precision_score(yi,p[:,i])
            aucs.append(auc);aps.append(ap)
            ra.plot(fpr,tpr,label=f'{short(name)}: {auc:.4f}',lw=1.6)
            pa.plot(recall,precision,label=f'{short(name)}: {ap:.4f}',lw=1.6)
            self.table(base+'/curve_data/roc_'+slug(name),pd.DataFrame({'fpr':fpr,'tpr':tpr,'threshold':thr}))
            self.table(base+'/curve_data/pr_'+slug(name),pd.DataFrame({'recall':recall,'precision':precision,'threshold':np.r_[th,np.nan]}))
            curves.append({'class':name,'roc_auc_ovr':auc,'average_precision_ovr':ap,'prevalence':yi.mean()})
        self.table(base+'/class_probability_metrics',pd.DataFrame(curves))
        m['macro_roc_auc_ovr']=float(np.mean(aucs)) if len(aucs)==k else np.nan
        m['macro_average_precision_ovr']=float(np.mean(aps)) if len(aps)==k else np.nan
        ra.plot([0,1],[0,1],'--',color='#9AA5B1',lw=1);ra.set(xlabel='False positive rate',ylabel='True positive rate',xlim=(0,1),ylim=(0,1.02));ra.legend(loc='lower right',fontsize=8)
        pa.set(xlabel='Recall',ylabel='Precision',xlim=(0,1),ylim=(0,1.02));pa.legend(loc='lower left',fontsize=8)
        self.fig(base+'/roc_curves',rocfig,title+'\nOne-vs-rest ROC (legend: AUROC)')
        self.fig(base+'/precision_recall_curves',prfig,title+'\nOne-vs-rest precision–recall (legend: AP)','AP is average precision, not trapezoidal PR area. Binary labels retain their saved meaning.')
        bins=np.linspace(0,1,11);idx=np.minimum(np.digitize(conf,bins,right=False)-1,9);cal=[]
        for b in range(10):
            mask=idx==b
            cal.append({'bin_low':bins[b],'bin_high':bins[b+1],'n':int(mask.sum()),
                        'mean_confidence':float(conf[mask].mean()) if mask.any() else np.nan,
                        'accuracy':float(correct[mask].mean()) if mask.any() else np.nan})
        cal=pd.DataFrame(cal);v=cal[cal.n>0];m['ece_10_equal_width']=float(np.sum(v.n/n*np.abs(v.accuracy-v.mean_confidence)))
        self.table(base+'/calibration_bins',cal)
        fig,ax=plt.subplots(figsize=(6,5));ax.plot([0,1],[0,1],'--',color='#9AA5B1');ax.scatter(v.mean_confidence,v.accuracy,s=30+200*v.n/n)
        ax.plot(v.mean_confidence,v.accuracy,lw=1);ax.set(xlabel='Mean confidence in bin',ylabel='Observed accuracy in bin',xlim=(0,1),ylim=(0,1.03))
        self.fig(base+'/calibration',fig,title+'\nTop-label reliability',f'10 equal-width bins; empty bins omitted. ECE={m["ece_10_equal_width"]:.4f}. No calibration fitted.')
        fig,ax=plt.subplots(figsize=(7,4.5))
        for mask,label,color in [(correct,'Correct',COLORS[0]),(~correct,'Incorrect',COLORS[1])]:
            ax.hist(conf[mask],bins=np.linspace(0,1,26),alpha=.65,label=f'{label} (n={mask.sum()})',color=color)
        ax.set(xlabel='Maximum predicted probability',ylabel='Images');ax.legend()
        self.fig(base+'/confidence_distribution',fig,title+'\nConfidence of correct and incorrect predictions')
        entropy=-np.sum(p*np.log(np.clip(p,1e-15,1)),axis=1)/np.log(k)
        fig,ax=plt.subplots(figsize=(7,4.5))
        ax.hist([entropy[correct],entropy[~correct]],bins=np.linspace(0,1,21),label=['Correct','Incorrect'],color=COLORS[:2]);ax.legend();ax.set(xlabel='Entropy / log(number of classes)',ylabel='Images')
        self.fig(base+'/entropy_distribution',fig,title+'\nPredictive uncertainty')
        # Threshold endpoints include all tied confidences; do not order ties arbitrarily.
        order=np.argsort(-conf,kind='stable');cs=conf[order];ends=np.r_[np.where(np.diff(cs)!=0)[0],n-1]
        covered=ends+1;acc=np.cumsum(correct[order])[ends]/covered
        risk=pd.DataFrame({'coverage':covered/n,'threshold':cs[ends],'accuracy':acc,'risk':1-acc})
        self.table(base+'/risk_coverage_data',risk)
        fig,ax=plt.subplots(figsize=(7,4.5));ax.plot(100*risk.coverage,100*risk.risk);ax.set(xlabel='Coverage: retained images (%)',ylabel='Error rate among retained images (%)')
        self.fig(base+'/risk_coverage',fig,title+'\nRisk–coverage curve','Retrospective diagnostic only. Do not choose a deployment threshold from the test curve.')
        meanp=np.vstack([p[y==i].mean(0) if np.any(y==i) else np.full(k,np.nan) for i in range(k)])
        self.heat(base+'/mean_probability_matrix',meanp*100,names,names,title+'\nMean predicted probabilities (%)',vmin=0,vmax=100,note='Rows: true class. Columns: probability assigned to each class.')
    def comparisons(self):
        if not self.metrics:return
        allm=pd.DataFrame(self.metrics);self.table('metrics_all_runs',allm)
        if allm.duplicated(['split','arm','seed']).any():raise ValueError('Multiple results for the same arm/seed/split; provide one experiment root only.')
        metriccols=['accuracy','macro_f1','balanced_accuracy','macro_precision','macro_recall','weighted_f1','mcc','cohen_kappa','log_loss','brier_multiclass_sum','ece_10_equal_width','macro_roc_auc_ovr','macro_average_precision_ovr']
        summaries=[]
        for (split,arm),g in allm.groupby(['split','arm'],sort=False):
            rr=[r for r in self.runs if r['split']==split and r['arm']==arm]
            ids=[set(r['df'].sample_id.astype(str)) for r in rr]
            if not all(s==ids[0] for s in ids):raise ValueError(f'{arm}/{split}: seed evaluations use different images.')
            for c in metriccols:
                if c not in g:continue
                a=g[c].dropna()
                summaries.append({'split':split,'arm':arm,'metric':c,'mean':a.mean(),'seed_sd':a.std(ddof=1) if len(a)>1 else np.nan,
                                  'n_seeds':len(a),'unique_images':len(ids[0]),'aggregation':'Mean of single-model runs; not an ensemble'})
        summary=pd.DataFrame(summaries);self.table('metrics_seed_summary',summary)
        for split in allm.split.unique():
            g=allm[allm.split==split];arms=list(g.arm.unique())
            for metric in ['accuracy','macro_f1','balanced_accuracy','mean_extra_crops']:
                if metric not in g:continue
                a=g.groupby('arm',sort=False)[metric].agg(['mean','std','count']).reindex(arms)
                factor=1 if metric=='mean_extra_crops' else 100
                fig,ax=plt.subplots(figsize=(max(7,len(arms)*1.15),4.8));x=np.arange(len(arms))
                ax.bar(x,a['mean']*factor,color=COLORS[0],alpha=.75)
                for i,arm in enumerate(arms):
                    values=g[g.arm==arm][metric].dropna().to_numpy()*factor
                    ax.scatter(np.full(len(values),i)+np.linspace(-.08,.08,len(values)),values,color='#16324F',s=25,zorder=4)
                    if len(values)>1:ax.errorbar(i,values.mean(),yerr=values.std(ddof=1),color='#16324F',capsize=4)
                    ax.text(i,values.mean()+(.8 if factor==100 else .025),f'{values.mean():.2f}',ha='center',fontsize=9)
                ax.set_xticks(x,[wrap(ARM_NAMES.get(a,a),17) for a in arms]);ax.set_ylabel(metric.replace('_',' ')+(' (%)' if factor==100 else ''))
                if factor==100:ax.set_ylim(0,106)
                self.fig(f'comparison/{split}_{metric}',fig,f'{self.name} | {split} comparison','Dots: individual seeds. Error bars: sample seed SD when n≥2; no CI or ensemble implied.')
            # Paired outcomes: never confuse equal aggregate accuracy with identical errors.
            rr=[r for r in self.runs if r['split']==split]
            pairrows=[]
            for seed in sorted(set(r['seed'] for r in rr)):
                sr=[r for r in rr if r['seed']==seed]
                for a,b in itertools.combinations(sr,2):
                    aa=a['df'].set_index('sample_id');bb=b['df'].set_index('sample_id')
                    if set(aa.index)!=set(bb.index):self.note('paired comparisons',f'{a["arm"]}/{b["arm"]}: IDs differ; skipped.');continue
                    bb=bb.loc[aa.index]
                    if not np.array_equal(aa.label,bb.label):raise ValueError('Aligned labels differ across methods.')
                    ac=aa.prediction==aa.label;bc=bb.prediction==bb.label
                    pairrows.append({'seed':seed,'method_a':a['arm'],'method_b':b['arm'],'n':len(aa),
                      'both_correct':int((ac&bc).sum()),'a_only_correct':int((ac&~bc).sum()),'b_only_correct':int((~ac&bc).sum()),
                      'both_wrong':int((~ac&~bc).sum()),'prediction_disagreements':int((aa.prediction!=bb.prediction).sum()),
                      'accuracy_gain_b_minus_a_pp':100*float(bc.mean()-ac.mean())})
            pairs=pd.DataFrame(pairrows);self.table(f'comparison/{split}_paired_outcomes',pairs)
            if len(pairs):
                chosen=pairs[pairs.method_a.isin(['GLOBAL','A_PLAIN'])|pairs.method_b.isin(['GLOBAL','A_PLAIN'])].copy()
                if len(chosen):
                    labels=chosen.method_a+' vs '+chosen.method_b+' / seed '+chosen.seed.astype(str)
                    fig,ax=plt.subplots(figsize=(9,max(4,len(chosen)*.35)));y=np.arange(len(chosen))
                    ax.barh(y,-chosen.a_only_correct,label='Only method A correct',color=COLORS[1]);ax.barh(y,chosen.b_only_correct,label='Only method B correct',color=COLORS[0])
                    ax.set_yticks(y,labels);ax.axvline(0,color='#16324F',lw=.7);ax.set_xlabel('Number of discordant predictions (A left; B right)');ax.legend(fontsize=8)
                    self.fig(f'comparison/{split}_paired_discordance',fig,f'{self.name} | paired errors', 'Descriptive paired counts; no independent-image significance test is applied to grouped data.')
        # Val and test are different held-out sets, connected lines would suggest a time curve.
        pivot=allm.groupby(['arm','split'])[['accuracy','macro_f1']].mean()
        for metric in ['accuracy','macro_f1']:
            t=pivot[metric].unstack('split')
            if len(t.columns)>1:
                fig,ax=plt.subplots(figsize=(9,5));(100*t).rename(index=ARM_NAMES).plot.barh(ax=ax,color=COLORS[:len(t.columns)]);ax.set_xlabel(metric.replace('_',' ')+' (%)');ax.set_ylabel('');ax.set_xlim(0,104)
                self.fig('comparison/validation_test_'+metric,fig,f'{self.name} | validation and final test','Different evaluation sets; this is not a test learning curve. No model is selected from this comparison.')
    def saved_statistics_and_efficiency(self):
        deployment=[]
        for p in report_files(self.root,'deployment.json'):
            row=readj(p);row['source']=str(p.relative_to(self.root));deployment.append(row)
        if deployment:self.table('efficiency/deployment_metadata',pd.DataFrame(deployment))
        for p in report_files(self.root,'test_group_bootstrap_ci.csv'):
            df=pd.read_csv(p)
            if not {'arm','seed','metric','estimate','ci95_low','ci95_high'}.issubset(df):continue
            for (metric,seed),t in df.groupby(['metric','seed'],sort=False):
                fig,ax=plt.subplots(figsize=(8,4));y=np.arange(len(t));ax.hlines(y,100*t.ci95_low,100*t.ci95_high,color=COLORS[0]);ax.scatter(100*t.estimate,y,color=COLORS[0]);ax.set_yticks(y,t.arm);ax.set_xlabel(metric+' (%)')
                self.fig(f'uncertainty/{slug(metric)}_seed_{slug(seed)}',fig,f'{self.name} | saved 95% bootstrap interval',
                    f'Seed/aggregation: {seed}. Conditional on fitted models; inspect saved uncertainty_protocol.json. This is not seed SD.')
        for p in report_files(self.root,'paired_effects.csv'):
            df=pd.read_csv(p)
            low=next((c for c in df if c in ('ci95_low_pp','ci_low_pp')),None);high=next((c for c in df if c in ('ci95_high_pp','ci_high_pp')),None)
            estimate=next((c for c in df if c in ('difference_pp','mean_difference_pp','estimate_pp')),None)
            if low and high and estimate and {'metric','contrast'}.issubset(df):
                for metric,t in df.groupby('metric'):
                    fig,ax=plt.subplots(figsize=(9,5));y=np.arange(len(t));ax.hlines(y,t[low],t[high],color=COLORS[0]);ax.scatter(t[estimate],y);ax.axvline(0,ls='--',color='#7B8794');ax.set_yticks(y,t.contrast);ax.set_xlabel('Difference (percentage points)')
                    self.fig('uncertainty/paired_'+slug(metric),fig,f'{self.name} | {metric} paired effects','Saved exploratory paired cluster intervals; not multiplicity-adjusted.')
        for p in report_files(self.root,'online_latency.csv'):
            df=pd.read_csv(p)
            if {'method','mean_ms','median_ms','p95_ms'}.issubset(df):
                fig,ax=plt.subplots(figsize=(8,4.5));df.set_index('method')[['mean_ms','median_ms','p95_ms']].plot.barh(ax=ax,color=COLORS[:3]);ax.set(xlabel='Milliseconds per image',ylabel='')
                self.fig('efficiency/'+slug(p.parent.relative_to(self.root))+'_latency',fig,f'{self.name} | saved inference latency',str(df.scope.iloc[0]) if 'scope' in df else 'Refer to source timing protocol.')
        for p in report_files(self.root,'efficiency_by_run.csv'):
            df=pd.read_csv(p)
            if 'arm' not in df:continue
            for c,scale,label in [('parameters',1e-6,'Parameters (millions)'),('checkpoint_mb',1,'Checkpoint size (MB)'),('counted_matrix_gmacs',1,'Counted matrix GMACs'),('forward_median_ms',1,'Forward median (ms)'),('decoded_to_logits_mean_ms',1,'Decoded-image to logits mean (ms)')]:
                if c not in df:continue
                t=df.groupby('arm')[c].mean()*scale;fig,ax=plt.subplots(figsize=(8,4.5));ax.barh(t.index,t.values);ax.set_xlabel(label)
                self.fig('efficiency/'+c,fig,f'{self.name} | {label}','Measured values copied from saved reports. Hardware/protocol remain in source_tables; MACs are not total FLOPs.')
            if 'forward_median_ms' in df and self.metrics:
                acc=pd.DataFrame(self.metrics).query("split == 'test'").groupby('arm').accuracy.mean()
                lat=df.groupby('arm').forward_median_ms.mean();common=acc.index.intersection(lat.index)
                if len(common):
                    fig,ax=plt.subplots(figsize=(7,4.5));ax.scatter(lat.loc[common],100*acc.loc[common])
                    for arm in common:ax.annotate(arm,(lat[arm],100*acc[arm]),xytext=(5,5),textcoords='offset points')
                    ax.set(xlabel='Mean of seed median forward latencies (ms)',ylabel='Mean single-model test accuracy (%)')
                    self.fig('efficiency/accuracy_latency',fig,f'{self.name} | accuracy and latency','Descriptive; no selection from test accuracy.')
    def finish(self):
        self.table('plot_index',pd.DataFrame(self.index));self.table('availability_and_notes',pd.DataFrame(self.notes))
        cards=[]
        for item in self.index:
            png=item['plot']+'.png';formats=' '.join(f'<a href="{html.escape(item["plot"]+"."+e)}">{e.upper()}</a>' for e in self.formats)
            cards.append(f'<section><h2>{html.escape(item["title"])}</h2><img loading="lazy" src="{html.escape(png)}"><p>{html.escape(item["note"])}</p>{formats}</section>')
        page='<!doctype html><meta charset="utf-8"><title>'+html.escape(self.name)+' visualizations</title><style>body{font:16px system-ui;margin:36px;background:#f2f5f7;color:#17324b}section{background:white;padding:22px;margin:24px 0;border-radius:12px}img{max-width:100%;max-height:700px}a{margin-right:14px}h2{font-size:18px}</style><h1>'+html.escape(self.name)+' — saved-result gallery</h1><p>See plot_index.csv, metrics_all_runs.csv, availability_and_notes.csv and source_tables for exact provenance. Extract the ZIP before opening this gallery.</p>'+''.join(cards)
        (self.out/'gallery.html').write_text(page)
        (self.out/'README.txt').write_text('LightCyto saved-result visualization export\n\n'
          'No training, threshold optimization or test inference was performed. No result values are invented.\n'
          'Unprefixed accuracy/macro_f1/ce in LightCyto history.csv are VALIDATION metrics. Training losses may include label smoothing and KD; they are not the same objective as validation CE.\n'
          'Final test has ROC and precision-recall curves, not a fabricated epoch-wise test curve.\n'
          'Seed SD is computed only for >=2 seeds. Seed means are not ensembles. Images are not multiplied by seed count.\n'
          'Sensitivity/recall and specificity use one-vs-rest class definitions; CRIC class 0/1 are not assigned a clinical meaning.\n'
          'Brier multiclass score sums squared error across classes; binary score uses probability of label 1. ECE uses 10 equal-width bins.\n'
          'Missing classes yield unavailable AUROC; undefined normalized confusion entries are NA. F1 uses zero_division=0.\n'
          'All test diagnostics are retrospective. Do not use them to retune the model or choose thresholds.\n'
          'Audit provenance and prior test use remain in source_tables. Different datasets/classes/splits are not a fair accuracy league table.\n'
          'Attention examples and architecture images, when present, are copied under saved_visuals. They are not newly computed explanations.\n'
          'New Grad-CAM, feature t-SNE/UMAP, raw image grids and robustness sweeps require images/features/checkpoints and are not fabricated from probabilities.\n')
        return {'dataset':self.name,'plots':len(self.index),'evaluation_runs':len(self.metrics),
                'validation_evaluation_runs':sum(m['split']=='validation' for m in self.metrics),
                'test_evaluation_runs':sum(m['split']=='test' for m in self.metrics),'path':str(self.out)}
    def run(self):
        self.copy_metadata();self.distribution();self.histories();self.history_comparisons()
        for p in sorted(report_files(self.root,'predictions.csv')):self.prediction_plots(p)
        self.comparisons();self.saved_statistics_and_efficiency()
        if not self.metrics:self.note('evaluation','No supported predictions.csv files found: confusion/ROC/PR require per-image predictions, not only screenshots or final accuracies.')
        self.note('advanced visualizations','New Grad-CAM / attention overlays need model and images; feature embeddings need saved feature arrays. This report exporter does not infer these from class probabilities.')
        return self.finish()


### Input handling and ZIP export

In [ ]:

def resolve_source(source,cache):
    p=Path(source).expanduser().resolve()
    if not p.exists():raise FileNotFoundError(f'Reports path not found: {p}')
    if p.is_file():
        if not zipfile.is_zipfile(p):raise ValueError('Use a reports ZIP or output directory, not an ipynb or checkpoint file.')
        target=Path(cache)/sha(p)[:16];target.mkdir(parents=True,exist_ok=True)
        with zipfile.ZipFile(p) as z:
            allowed=[i for i in z.infolist() if Path(i.filename).suffix.lower() in ('.csv','.json','.png','.svg','.pdf')]
            if sum(i.file_size for i in allowed)>4*1024**3:raise ValueError('Reports exceed 4 GiB; supply the small reports archive.')
            for info in allowed:
                dest=(target/info.filename).resolve()
                if not dest.is_relative_to(target.resolve()):raise ValueError('Unsafe ZIP member path.')
                dest.parent.mkdir(parents=True,exist_ok=True)
                with z.open(info) as src,open(dest,'wb') as dst:shutil.copyfileobj(src,dst)
        p=target
    markers=list(CONTRACT_MARKERS)
    # An explicit experiment root wins over an embedded reports copy.
    if any((p/marker).is_file() for marker in markers):return p
    roots=sorted(set(f.parent for marker in markers for f in p.rglob(marker)))
    roots=[r for r in roots if not any(a!=r and a in r.parents for a in roots)]
    if len(roots)==1:return roots[0]
    if len(roots)>1:raise ValueError(f'Multiple experiments in {p}. Set the exact single-experiment output directory.')
    return p

def discover_sources(search_roots):
    """List candidates; never silently choose between different saved versions."""
    rows=[]
    for root in search_roots:
        root=Path(root)
        if not root.exists():continue
        for p in root.rglob('*'):
            if p.is_file() and ((p.suffix=='.zip' and 'report' in p.name.lower() and 'lightcyto' in p.name.lower()) or p.name in ('experiment.json','ablation_contract.json')):
                rows.append({'path':str(p.parent if p.suffix=='.json' else p),'type':'folder' if p.suffix=='.json' else 'zip'})
    return pd.DataFrame(rows).drop_duplicates() if rows else pd.DataFrame(columns=['path','type'])

def export_all(sources,output_root,formats=('png','pdf','svg'),dpi=300):
    if 'png' not in formats:raise ValueError('PNG is required for the HTML gallery; optionally add PDF/SVG.')
    out=Path(output_root).expanduser().resolve();out.mkdir(parents=True,exist_ok=True)
    # Each invocation gets its own folder; stale plots cannot leak from earlier runs.
    from datetime import datetime, timezone
    tag=datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f')
    session=out/('export_'+tag);session.mkdir();statuses=[];zips=[];resolved=[]
    for name,source in sources.items():
        if not str(source).strip():
            statuses.append({'dataset':name,'status':'missing_input','detail':'Set this dataset report ZIP or full output folder path.'});continue
        root=resolve_source(source,out/'_extracted');resolved.append(str(root))
        if len(set(resolved))!=len(resolved):raise ValueError('The same report source is assigned to multiple datasets.')
        print(f'Exporting {name}: {root}',flush=True)
        ex=Exporter(name,root,session/slug(name),formats,dpi);status=ex.run();status['status']='complete' if status['test_evaluation_runs'] else 'partial_no_test' if status['evaluation_runs'] else 'partial_no_predictions';statuses.append(status)
        zips.append(shutil.make_archive(str(out/(slug(name)+'_Visualizations_'+tag)),'zip',root_dir=ex.out))
        print(f'{name}: {status["plots"]} figures, {status["evaluation_runs"]} evaluation runs.',flush=True)
    writej(session/'dataset_status.json',statuses)
    writej(session/'export_protocol.json',{'version':VERSION,'created_utc':tag,'formats':formats,'dpi':dpi,'sources':{k:str(v) for k,v in sources.items()},
        'no_training':True,'no_new_test_inference':True,'cross_dataset_ranking':False})
    (session/'README.txt').write_text('Open each dataset/gallery.html after extracting. This bundle includes only supplied results. See dataset_status.json for missing datasets. No fabricated results or synthetic test fixtures are included.\n')
    master=shutil.make_archive(str(out/('LightCyto_All_Visualizations_'+tag)),'zip',root_dir=session)
    return {'folder':str(session),'combined_zip':master,'dataset_zips':zips,'status':statuses}

def configured_sources(sources, folder_names, search_roots, cache):
    """Resolve the screenshot folders; reject missing/ambiguous AUTO inputs before plotting."""
    requested={key:folder_names[key] for key,value in sources.items() if str(value).strip().upper() in ('AUTO','')}
    hits={key:set() for key in requested}
    reverse={name:key for key,name in requested.items()}
    for search_root in search_roots:
        search_root=Path(search_root)
        if not search_root.exists():continue
        for current,dirs,_ in os.walk(search_root,followlinks=False):
            current=Path(current)
            if current.name in reverse:
                hits[reverse[current.name]].add(current.resolve())
                dirs[:]=[]  # Do not descend into reports copies of this output.
            else:
                dirs[:]=[d for d in dirs if not d.startswith('.') and d!='__MACOSX']
    resolved={};problems=[]
    for name,value in sources.items():
        text=str(value).strip()
        if text.upper()=='SKIP':resolved[name]='';continue
        if name in requested:
            candidates=sorted(hits[name])
            if not candidates:
                problems.append(f"{name}: could not find folder '{requested[name]}' under {[str(p) for p in search_roots]}. Attach that saved output or set SOURCES['{name}'] to its exact ZIP/folder path.")
                continue
            if len(candidates)>1:
                problems.append(f"{name}: multiple saved versions found. Set SOURCES['{name}'] explicitly: " + ' | '.join(map(str,candidates)))
                continue
            value=candidates[0]
        try:resolved[name]=str(resolve_source(value,cache))
        except (ValueError,FileNotFoundError) as error:problems.append(f'{name}: {error}')
    if problems:raise ValueError('Input resolution stopped before generating plots.\n\n'+'\n\n'.join(problems))
    used=[p for p in resolved.values() if p]
    if len(set(used))!=len(used):raise ValueError('The same experiment was assigned to two datasets.')
    rows=[]
    for name,source in resolved.items():
        if not source:
            rows.append({'dataset':name,'source':'SKIP','training_histories':0,'validation_predictions':0,'test_predictions':0,'status':'skipped'});continue
        root=Path(source)
        if not any((root/m).is_file() for m in CONTRACT_MARKERS):
            raise ValueError(f'{name}: {root} is not a verified LightCyto experiment root; experiment.json or ablation_contract.json is required.')
        preds=list(report_files(root,'predictions.csv'));nval=0;ntest=0
        for p in preds:
            parts=p.relative_to(root).parts
            if 'final_test' in parts or 'test' in parts:ntest+=1
            elif 'validation' in parts or 'val' in parts:nval+=1
        histories=len(list(report_files(root,'history.csv')))
        if not histories and not preds:
            raise ValueError(f'{name}: no history.csv or predictions.csv found in {root}. Attach reports/full output, not weights alone.')
        row={'dataset':name,'source':source,'training_histories':histories,'validation_predictions':nval,'test_predictions':ntest,
             'status':'test available' if ntest else 'training/validation only; test predictions missing'}
        rows.append(row)
        if not ntest:print(f'{name}: no saved final-test predictions. Available training/validation plots will be exported; no new test inference runs.',flush=True)
    return resolved,pd.DataFrame(rows)


## 4. Resolve and check your three saved outputs

This cell fills the exact mounted paths automatically and displays the number of history and prediction files actually found. A missing/ambiguous input stops here with the required folder name, before spending time generating hundreds of figures.

Nested `lightcyto_*_reports` copies are excluded when the full output folder is selected, so their duplicated histories/predictions are not counted twice. Missing **test predictions** do not block training/validation plots; they are explicitly reported.


In [ ]:
from IPython.display import display, FileLink, Markdown
RESOLVED_SOURCES, INPUT_CHECK = configured_sources(
    SOURCES, SOURCE_FOLDER_NAMES, SEARCH_ROOTS, OUTPUT_ROOT / '_extracted'
)
display(INPUT_CHECK)
print('Resolved paths:')
for dataset, source in RESOLVED_SOURCES.items():
    print(f'{dataset}: {source or "SKIP"}')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
writej(OUTPUT_ROOT / 'resolved_sources.json', RESOLVED_SOURCES)
INPUT_CHECK.to_csv(OUTPUT_ROOT / 'input_check.csv', index=False)


## 5. Generate all available visualizations

This reads saved CSV/JSON reports. Runtime depends on how many runs and export formats are present. It prints progress by dataset and never starts model training. A fresh export directory prevents figures from previous runs being accidentally mixed in.

If validation fails (for example class-probability order or sample IDs disagree), resolve the source report mismatch instead of bypassing the check.


In [ ]:
if not any(RESOLVED_SOURCES.values()):
    raise ValueError('All datasets were set to SKIP. Enable at least one source.')
EXPORT = export_all(RESOLVED_SOURCES, OUTPUT_ROOT, formats=FORMATS, dpi=DPI)
display(pd.DataFrame(EXPORT['status']))
print('Export directory:', EXPORT['folder'])


## 6. Download everything

The first link is the **combined ZIP**. Separate dataset ZIPs are also provided. If a notebook frontend does not open a link, download the corresponding ZIP from Kaggle's Output/Files panel under `lightcyto_visualizations`.

Extract the ZIP locally and open `DATASET/gallery.html` to browse. Use PNG in slides/Word, PDF in a paper, and SVG for vector editing. CSV files contain exact values and curve coordinates.


In [ ]:
display(Markdown('### Combined download'))
for filename in [EXPORT['combined_zip'], *EXPORT['dataset_zips']]:
    display(FileLink(os.path.relpath(filename, Path.cwd())))
print('Alternatively download these ZIPs from Kaggle Output → lightcyto_visualizations.')
print('See dataset_status.json for missing test evaluations; no placeholder results are invented.')


## 7. Review the result tables and provenance

For a three-seed experiment, the summary is the mean of **three individual models**, not an ensemble. A single-seed experiment has no estimate of seed SD. Test bootstrap CIs and seed SD represent different sources of uncertainty.

The binary CRIC repack's Class_0/Class_1 names remain unchanged. Do not infer a clinical diagnosis from their numeric code. Source/patient independence depends on `data_audit.json`, not on the graph title.

Different datasets have different labels, sample counts and split units. We produce separate dataset reports rather than ranking their accuracy as though the tasks were identical. Existing repeated/post-selection test use remains part of the study's scope; plotting cannot make it an untouched external test.


In [ ]:
for item in EXPORT['status']:
    if item.get('status') == 'missing_input':
        print(item['dataset'], ': result path not supplied')
        continue
    folder = Path(item['path'])
    print('Dataset:', item['dataset'], '|', item['plots'], 'figures')
    for filename in ['metrics_seed_summary.csv', 'availability_and_notes.csv']:
        path = folder / filename
        if path.exists():
            display(pd.read_csv(path))
